# Wildfire Prediction — Comprehensive Flow Notebook

This notebook consolidates preliminary analysis, EDA, preprocessing, classifier training, regressor training, and clustering into a single end-to-end flow.

**Sections:**
1. Preliminary Analysis
2. Exploratory Data Analysis (EDA)
3. Preprocessing
4. Classifier Training and Results
5. Regressor Training and Results
6. Clustering Training and Results


## 1. Preliminary Analysis

### 1.1 Setup — Libraries and Visual Style

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    mean_absolute_error,
    r2_score,
    silhouette_score,
)
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from scipy.stats import spearmanr

try:
    from imblearn.over_sampling import SMOTE
    _SMOTE_AVAILABLE = True
except ImportError:
    _SMOTE_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    _LGBM_AVAILABLE = True
except ImportError:
    _LGBM_AVAILABLE = False
    print("WARNING: LightGBM not installed. Regressor section will be skipped.")

try:
    import joblib
except ImportError:
    joblib = None

warnings.filterwarnings("ignore")

# ── Consistent visual style ──────────────────────────────────────────────────
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ── Shared colour palette ─────────────────────────────────────────────────────
COLOR_FIRE    = "#B85042"
COLOR_NOFIRE  = "#5D737E"
COLOR_DAY     = "#E8A33D"
COLOR_NIGHT   = "#1E2761"

RANDOM_STATE        = 42
# Minimum minority-class proportion below which SMOTE is applied.
# 0.40 means a class must hold at least 40 % of samples for the
# dataset to be considered balanced enough to skip oversampling.
IMBALANCE_THRESHOLD = 0.40

print("All imports loaded successfully.")


### 1.2 Load Dataset

In [ ]:
repo_root = Path.cwd()
candidates = [repo_root / "wildfire.csv", repo_root / "final_dataset.csv"]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find wildfire.csv or final_dataset.csv in the working directory."
    )

df = pd.read_csv(data_path)
print(f"Loaded  : {data_path.name}")
print(f"Shape   : {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


### 1.3 Dataset Structure

In [ ]:
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")
print()
print(df.dtypes)


## 2. Exploratory Data Analysis (EDA)

### 2.1 Dataset Health Check

#### Missing Values

In [ ]:
missing = df.isna().sum()
total_missing = missing.sum()

if total_missing == 0:
    print("No missing values across the entire dataset.")
else:
    print(f"Total missing cells: {total_missing}")
    print(missing[missing > 0])


#### Duplicate Rows

In [ ]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")


### 2.2 Target Variable Analysis

The pipeline has two supervised targets:
- `occured` — binary fire occurrence (classification)
- `frp` — Fire Radiative Power in megawatts (regression)

#### Classification Target: `occured`

In [ ]:
counts = df["occured"].value_counts().sort_index()
proportions = df["occured"].value_counts(normalize=True).sort_index().round(4)

print("Counts:")
print(counts)
print("\nProportions:")
print(proportions)

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(["No Fire (0)", "Fire (1)"], counts.values,
              color=[COLOR_NOFIRE, COLOR_FIRE])
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v,
            f"{v:,}\n({v / len(df) * 100:.1f}%)",
            ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("Class Balance: Fire Occurrence", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of observations")
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.show()


#### Regression Target: `frp` (Fire Radiative Power)

In [ ]:
print(df["frp"].describe())
print(f"\nSkewness: {df['frp'].skew():.2f}")
print(f"Rows with FRP == 0: {(df['frp'] == 0).sum()}")
print(f"Rows with FRP  > 0: {(df['frp'] > 0).sum():,}")


In [ ]:
print("FRP statistics grouped by occured:")
df.groupby("occured")["frp"].describe().round(2)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fire_frp = df[df["occured"] == 1]["frp"]

axes[0].hist(fire_frp, bins=80, color=COLOR_FIRE, edgecolor="white")
axes[0].set_title("FRP Distribution (raw) — fire events only", fontweight="bold")
axes[0].set_xlabel("FRP (MW)")
axes[0].set_ylabel("Frequency")

axes[1].hist(np.log1p(fire_frp), bins=60, color=COLOR_FIRE, edgecolor="white")
axes[1].set_title("FRP Distribution (log1p) — fire events only", fontweight="bold")
axes[1].set_xlabel("log(FRP + 1)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


### 2.3 Feature Analysis

#### Geographic Distribution

In [ ]:
print(f"Latitude range:  {df['lat'].min():.2f} to {df['lat'].max():.2f}")
print(f"Longitude range: {df['lon'].min():.2f} to {df['lon'].max():.2f}")

fig, ax = plt.subplots(figsize=(12, 6))

no_fire = df[df["occured"] == 0].sample(
    min(5000, (df["occured"] == 0).sum()), random_state=RANDOM_STATE)
fire = df[df["occured"] == 1]

ax.scatter(no_fire["lon"], no_fire["lat"], s=2, c=COLOR_NOFIRE,
           alpha=0.3, label="No fire (5k sample)")
ax.scatter(fire["lon"], fire["lat"], s=3, c=COLOR_FIRE,
           alpha=0.5, label="Fire")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Global distribution of observations", fontweight="bold")
ax.legend(markerscale=4)
plt.tight_layout()
plt.show()


#### Day vs. Night Observations

In [ ]:
dn = df.groupby("daynight_N")["occured"].agg(["count", "sum", "mean"])
dn["fire_rate_pct"] = dn["mean"] * 100
print(dn)

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(["Day (0)", "Night (1)"], dn["fire_rate_pct"].values,
              color=[COLOR_DAY, COLOR_NIGHT])
for b, v in zip(bars, dn["fire_rate_pct"].values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%",
            ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_title("Fire occurrence rate: Day vs Night", fontweight="bold")
ax.set_ylabel("Fire rate (%)")
ax.set_ylim(0, dn["fire_rate_pct"].max() * 1.2)
plt.tight_layout()
plt.show()


#### Feature Distributions: Fire vs. No-Fire

In [ ]:
features_to_plot = [
    "fire_weather_index", "temp_mean", "humidity_min",
    "wind_speed_max", "solar_radiation_mean", "evapotranspiration_total",
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flatten(), features_to_plot):
    data_nofire = df[df["occured"] == 0][feat].dropna()
    data_fire   = df[df["occured"] == 1][feat].dropna()
    ax.hist(data_nofire, bins=40, alpha=0.55, color=COLOR_NOFIRE,
            label="No fire", density=True)
    ax.hist(data_fire,   bins=40, alpha=0.55, color=COLOR_FIRE,
            label="Fire",    density=True)
    ax.set_title(feat, fontweight="bold")
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.suptitle("Feature distributions: Fire vs. No-Fire", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


#### Feature Means: Fire vs. No-Fire

In [ ]:
feat_cols = [
    "fire_weather_index", "temp_mean", "humidity_min",
    "wind_speed_max", "solar_radiation_mean", "cloud_cover_mean",
    "dewpoint_mean", "evapotranspiration_total", "temp_range",
]
df.groupby("occured")[feat_cols].mean().round(2).T


### 2.4 Correlation Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.75}, ax=ax,
            annot_kws={"size": 8})
ax.set_title("Feature Correlation Matrix", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()


#### Correlation with `occured`

In [ ]:
corr_occured = (
    df.drop(columns=["frp"])
      .corr(numeric_only=True)["occured"]
      .sort_values(key=abs, ascending=False)
)
print(corr_occured)


#### Correlation with `frp` (fire-only subset)

In [ ]:
fire_df = df[df["occured"] == 1]
corr_frp = (
    fire_df.drop(columns=["occured"])
           .corr(numeric_only=True)["frp"]
           .sort_values(key=abs, ascending=False)
)
print(corr_frp)
print(f"\nFire-only subset size: {len(fire_df):,}")


## 3. Preprocessing

### 3.1 Feature / Target Separation

For **classification** we use `occured` as the target and exclude `frp` (the regression target).
For **regression** we use fire-only rows and `frp` as the target.

In [ ]:
# ── Classification setup ──────────────────────────────────────────────────────
clf_df = df.dropna(subset=["occured"]).copy()

# log-transform frp for potential use in features
if "frp" in clf_df.columns and "frp_log" not in clf_df.columns:
    clf_df["frp_log"] = np.log1p(clf_df["frp"].clip(lower=0))

# Features = all numeric columns except the two targets
clf_feature_cols = [
    c for c in clf_df.select_dtypes(include=[np.number]).columns
    if c not in ("occured", "frp", "frp_log")
]
X_clf = clf_df[clf_feature_cols]
y_clf = clf_df["occured"]

print(f"Classification features : {clf_feature_cols}")
print(f"X shape                 : {X_clf.shape}")
print(f"Class balance:\n{y_clf.value_counts()}")


### 3.2 Train / Test Split

In [ ]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_clf,
)

print(f"Train size : {len(X_train_clf):,}")
print(f"Test size  : {len(X_test_clf):,}")


### 3.3 Oversampling (SMOTE)

The dataset is pre-balanced (50/50), so SMOTE is applied only if the training set becomes imbalanced after splitting.

In [ ]:
train_balance = y_train_clf.value_counts(normalize=True)
imbalanced = train_balance.min() < IMBALANCE_THRESHOLD

if _SMOTE_AVAILABLE and imbalanced:
    sm = SMOTE(random_state=RANDOM_STATE)
    X_res, y_res = sm.fit_resample(X_train_clf, y_train_clf)
    print(f"SMOTE applied. Resampled training size: {len(X_res):,}")
else:
    X_res, y_res = X_train_clf.copy(), y_train_clf.copy()
    if imbalanced:
        print("SMOTE not available; using original training split.")
    else:
        print(f"Dataset already balanced ({train_balance.to_dict()}); SMOTE skipped.")

print(f"Training class balance after resampling:\n{pd.Series(y_res).value_counts()}")


### 3.4 Feature Scaling

In [ ]:
scaler_clf = StandardScaler()
X_train_scaled = scaler_clf.fit_transform(X_res)
X_test_scaled  = scaler_clf.transform(X_test_clf)
print("StandardScaler fitted on training data.")


## 4. Classifier Training and Results

### 4.1 Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_res, y_res)
rf_preds = rf.predict(X_test_clf)

print("=== Random Forest ===")
print(classification_report(y_test_clf, rf_preds))
print(f"ROC-AUC: {roc_auc_score(y_test_clf, rf.predict_proba(X_test_clf)[:, 1]):.4f}")


#### Random Forest — Feature Importance

In [ ]:
fi_rf = pd.Series(rf.feature_importances_, index=clf_feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
fi_rf.head(15).sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 15 Feature Importances — Random Forest", fontweight="bold")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()


#### Random Forest — Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test_clf, rf_preds, ax=ax,
                                        display_labels=["No Fire", "Fire"],
                                        colorbar=False)
ax.set_title("Confusion Matrix — Random Forest", fontweight="bold")
plt.tight_layout()
plt.show()


### 4.2 Support Vector Machine (SVM)

In [ ]:
svm = SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
svm.fit(X_train_scaled, y_res)
svm_preds = svm.predict(X_test_scaled)

print("=== SVM (RBF kernel) ===")
print(classification_report(y_test_clf, svm_preds))
print(f"ROC-AUC: {roc_auc_score(y_test_clf, svm.predict_proba(X_test_scaled)[:, 1]):.4f}")


#### SVM — Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test_clf, svm_preds, ax=ax,
                                        display_labels=["No Fire", "Fire"],
                                        colorbar=False)
ax.set_title("Confusion Matrix — SVM", fontweight="bold")
plt.tight_layout()
plt.show()


### 4.3 Classifier Comparison Summary

In [ ]:
summary_data = {
    "Model": ["Random Forest", "SVM (RBF)"],
    "Accuracy": [
        (rf_preds == y_test_clf.values).mean(),
        (svm_preds == y_test_clf.values).mean(),
    ],
    "ROC-AUC": [
        roc_auc_score(y_test_clf, rf.predict_proba(X_test_clf)[:, 1]),
        roc_auc_score(y_test_clf, svm.predict_proba(X_test_scaled)[:, 1]),
    ],
}
pd.DataFrame(summary_data).set_index("Model").round(4)


## 5. Regressor Training and Results

We predict **fire intensity** (FRP) only for observations where a fire actually occurred. A `log1p` transformation is applied to handle the heavy right-skew of FRP.

### 5.1 Data Filtering and Preparation

In [ ]:
fire_only_df = df[df["occured"] == 1.0].copy()
fire_only_df = fire_only_df.drop(columns=["occured"])

X_reg = fire_only_df.drop(columns=["frp"])
y_reg = np.log1p(fire_only_df["frp"])

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Original dataset shape : {df.shape}")
print(f"Fire-only dataset shape: {fire_only_df.shape}")
print(f"Training set shape     : {X_train_reg.shape}")
print(f"Testing set shape      : {X_test_reg.shape}")


### 5.2 LightGBM Regressor Training

In [ ]:
if not _LGBM_AVAILABLE:
    print("LightGBM is not installed. Skipping regressor training.")
else:
    reg_model = LGBMRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=8,
        num_leaves=50,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )
    reg_model.fit(X_train_reg, y_train_reg)
    print("LightGBM regressor training complete.")


### 5.3 Evaluation and Insights

In [ ]:
if not _LGBM_AVAILABLE:
    print("Skipping — LightGBM not available.")
else:
    log_preds = reg_model.predict(X_test_reg)

    y_test_real  = np.expm1(y_test_reg)
    preds_real   = np.expm1(log_preds)

    r2  = r2_score(y_test_real, preds_real)
    mae = mean_absolute_error(y_test_real, preds_real)

    print(f"R-squared (R²): {r2:.4f}")
    print(f"Mean Absolute Error (MAE): {mae:.2f} MW")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Actual vs. Predicted
    cap = np.percentile(y_test_real, 95)
    axes[0].scatter(y_test_real, preds_real, alpha=0.3, color="teal", s=4)
    axes[0].plot([0, cap], [0, cap], "r--", linewidth=1.5)
    axes[0].set_xlim(0, cap)
    axes[0].set_ylim(0, np.percentile(preds_real, 95))
    axes[0].set_xlabel("Actual FRP (MW)")
    axes[0].set_ylabel("Predicted FRP (MW)")
    axes[0].set_title("Actual vs. Predicted Fire Intensity", fontweight="bold")
    axes[0].grid(True, linestyle="--", alpha=0.5)

    # Residual Plot
    residuals = y_test_real - preds_real
    axes[1].scatter(preds_real, residuals, alpha=0.3, color="darkred", s=4)
    axes[1].axhline(y=0, color="black", linestyle="--", linewidth=1.5)
    axes[1].set_xlim(0, np.percentile(preds_real, 95))
    axes[1].set_xlabel("Predicted FRP (MW)")
    axes[1].set_ylabel("Residuals (Actual − Predicted)")
    axes[1].set_title("Residual Plot", fontweight="bold")
    axes[1].grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()


### 5.4 Feature Importance

In [ ]:
if not _LGBM_AVAILABLE:
    print("Skipping — LightGBM not available.")
else:
    fi_df = pd.DataFrame({
        "Feature": X_train_reg.columns,
        "Importance": reg_model.feature_importances_,
    }).sort_values(by="Importance", ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(x="Importance", y="Feature", data=fi_df, palette="magma")
    plt.title("Feature Importance — LightGBM Regressor", fontweight="bold")
    plt.xlabel("Importance (tree splits)")
    plt.tight_layout()
    plt.show()


## 6. Clustering Training and Results

Unsupervised clustering on meteorological features only. `fire_weather_index`, `occured`, `frp`, `lat`, and `lon` are used only for **post-hoc validation** — never during fitting.

### 6.1 Setup and Feature Selection

In [ ]:
CLUSTER_FEATURES = [
    "pressure_mean", "wind_direction_mean", "wind_direction_std",
    "solar_radiation_mean", "dewpoint_mean", "cloud_cover_mean",
    "evapotranspiration_total", "humidity_min", "temp_mean",
    "temp_range", "wind_speed_max",
]
VALIDATION_FEATURES = ["fire_weather_index", "lat", "lon", "occured", "frp"]
RISK_TIER_NAMES = ["Low", "Mod", "High", "Extreme"]
geo_colors = ["#2166ac", "#67a9cf", "#fdae61", "#b2182b"]
N_CLUSTERS = 4
k_values = [2, 3, 4, 5, 6]

# Verify required columns exist
missing_clust = [c for c in CLUSTER_FEATURES if c not in df.columns]
if missing_clust:
    print(f"WARNING: Missing clustering columns: {missing_clust}")

X_clust = df[CLUSTER_FEATURES].copy()
X_val   = df[VALIDATION_FEATURES].copy()

scaler_clust = StandardScaler()
X_arr = scaler_clust.fit_transform(X_clust)
print(f"Clustering feature matrix shape : {X_arr.shape}")
print(f"Validation feature matrix shape : {X_val.shape}")


### 6.2 Helper Functions

In [ ]:
def remap_labels_by_fwi(frame: pd.DataFrame, label_col: str) -> dict:
    """Reorder cluster labels so label 0 = lowest mean FWI, 3 = highest."""
    means = frame.groupby(label_col)["fire_weather_index"].mean().sort_values()
    return {old: new for new, old in enumerate(means.index)}


def boxplot_fwi(frame: pd.DataFrame, label_col: str, title: str) -> None:
    ordered_labels = list(range(N_CLUSTERS))
    values = [
        frame.loc[frame[label_col] == lbl, "fire_weather_index"].to_numpy()
        for lbl in ordered_labels
    ]
    fig, ax = plt.subplots(figsize=(8, 5))
    bp = ax.boxplot(values, labels=RISK_TIER_NAMES, patch_artist=True, showfliers=False)
    box_colors = ["#dbe9f6", "#a6bddb", "#fdcc8a", "#e34a33"]
    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_edgecolor("black")
    for group in ("whiskers", "caps", "medians"):
        for artist in bp[group]:
            artist.set_color("black")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Risk tier")
    ax.set_ylabel("fire_weather_index")
    ax.grid(True, axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

print("Helper functions defined.")


### 6.3 K-Means Baseline

In [ ]:
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
kmeans_raw = kmeans.fit_predict(X_arr)

km_map = remap_labels_by_fwi(df.assign(kmeans_label=kmeans_raw), "kmeans_label")
df["kmeans_label"] = pd.Series(kmeans_raw, index=df.index).map(km_map).astype(int)

km_silhouette = silhouette_score(
    X_arr, df["kmeans_label"],
    sample_size=min(10_000, len(df)), random_state=RANDOM_STATE,
)
km_means = df.groupby("kmeans_label")["fire_weather_index"].mean().sort_index()

print(f"K-Means silhouette score (K={N_CLUSTERS}): {km_silhouette:.4f}")
print("K-Means mean fire_weather_index by cluster:")
print(km_means.rename("mean_fire_weather_index").to_frame())


#### K-Means — Silhouette Sweep

In [ ]:
sil_scores = []
for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(X_arr)
    sil_scores.append(
        silhouette_score(X_arr, labels,
                         sample_size=min(10_000, len(df)),
                         random_state=RANDOM_STATE)
    )

_score_k4 = sil_scores[k_values.index(4)]
_score_k2 = sil_scores[k_values.index(2)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, sil_scores, marker="o", linewidth=2, color="#1f77b4")
ax.axvline(x=4, color="red", linestyle="--", alpha=0.7)
ax.annotate("Domain-selected\n(4-tier convention)",
            xy=(4, _score_k4), xytext=(4.2, _score_k4 + 0.005),
            arrowprops=dict(arrowstyle="->"), fontsize=9)
ax.annotate("K=2: binary split",
            xy=(2, _score_k2), xytext=(2.2, _score_k2 - 0.008), fontsize=9)
ax.set_xticks(k_values)
ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("Silhouette score")
ax.set_title("Silhouette Sweep for K-Means", fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


#### K-Means — FWI Boxplot

In [ ]:
boxplot_fwi(df, "kmeans_label", f"FWI Distribution by Cluster (K-Means, K={N_CLUSTERS})")


### 6.4 Gaussian Mixture Model (GMM)

In [ ]:
gmm = GaussianMixture(n_components=N_CLUSTERS, covariance_type="full",
                      random_state=RANDOM_STATE)
gmm.fit(X_arr)
gmm_raw   = gmm.predict(X_arr)
gmm_proba = gmm.predict_proba(X_arr)

gmm_map = remap_labels_by_fwi(df.assign(gmm_label=gmm_raw), "gmm_label")
df["gmm_label"] = pd.Series(gmm_raw, index=df.index).map(gmm_map).astype(int)

gmm_means = df.groupby("gmm_label")["fire_weather_index"].mean().sort_index()
gmm_bic   = gmm.bic(X_arr)
gmm_aic   = gmm.aic(X_arr)

print(f"GMM BIC (K={N_CLUSTERS}): {gmm_bic:.2f}")
print(f"GMM AIC (K={N_CLUSTERS}): {gmm_aic:.2f}")
print("GMM mean fire_weather_index by cluster:")
print(gmm_means.rename("mean_fire_weather_index").to_frame())


#### GMM — BIC / AIC Sweep

In [ ]:
bic_scores = []
aic_scores = []
for k in k_values:
    model = GaussianMixture(n_components=k, covariance_type="full",
                            random_state=RANDOM_STATE)
    model.fit(X_arr)
    bic_scores.append(model.bic(X_arr))
    aic_scores.append(model.aic(X_arr))

_bic_k4 = bic_scores[k_values.index(4)]
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(k_values, bic_scores, marker="o", linewidth=2, color="#2ca02c", label="BIC")
ax.plot(k_values, aic_scores, marker="o", linewidth=2, color="#d62728", label="AIC")
ax.axvline(x=4, color="orange", linestyle="--", alpha=0.8, label="Domain-selected K")
ax.annotate("Marginal gain flattens near K=4",
            xy=(4, _bic_k4), xytext=(4.2, _bic_k4 + 5000),
            arrowprops=dict(arrowstyle="->"), fontsize=9)
ax.set_xticks(k_values)
ax.set_xlabel("Number of components")
ax.set_ylabel("Score (lower is better)")
ax.set_title("GMM BIC/AIC Sweep", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


#### GMM — FWI Boxplot

In [ ]:
boxplot_fwi(df, "gmm_label", f"FWI Distribution by Cluster (GMM, K={N_CLUSTERS})")


### 6.5 Post-Model Analysis and Comparison

In [ ]:
for label_col, model_name in [("kmeans_label", "K-Means"), ("gmm_label", "GMM")]:
    table = (
        df.groupby(label_col)["fire_weather_index"]
          .mean().sort_index()
          .to_frame("mean_fire_weather_index")
    )
    rho = spearmanr(table.index.to_numpy(),
                    table["mean_fire_weather_index"].to_numpy()).statistic

    print(f"=== {model_name}: mean FWI by cluster ===")
    print(table)
    print(f"Spearman ρ (cluster rank vs. mean FWI): {rho:.4f}\n")


In [ ]:
# Cluster profiles
cluster_profile_gmm    = df.groupby("gmm_label")[CLUSTER_FEATURES].mean().sort_index()
cluster_profile_kmeans = df.groupby("kmeans_label")[CLUSTER_FEATURES].mean().sort_index()

fire_table = df.groupby("gmm_label").agg(
    fire_occurrence_rate=("occured", "mean"),
    mean_frp_given_fire=("frp", lambda s: s[s > 0].mean()),
).sort_index()

cluster_summary = df.groupby("gmm_label").agg(
    size=("gmm_label", "count"),
    fire_rate=("occured", "mean"),
    median_frp=("frp", lambda s: s[s > 0].median()),
).sort_index()
min_med_frp = cluster_summary["median_frp"].min()
cluster_summary["zone_multiplier"] = (cluster_summary["median_frp"] / min_med_frp).round(2)

print("GMM cluster profile (feature means):")
print(cluster_profile_gmm.round(2))
print("\nFire occurrence and FRP by GMM cluster:")
print(fire_table.round(3))
print("\nCluster summary (GMM):")
print(cluster_summary.round(3))


### 6.6 Visualization Suite

#### Standardized Cluster Profile Heatmap (GMM)

In [ ]:
gmm_profile_z = (
    (cluster_profile_gmm - cluster_profile_gmm.mean(axis=0))
    / cluster_profile_gmm.std(axis=0, ddof=0)
)

fig, ax = plt.subplots(figsize=(14, 4.5))
img = ax.imshow(gmm_profile_z.to_numpy(), cmap="RdYlBu_r", aspect="auto")
for row_idx in range(gmm_profile_z.shape[0]):
    for col_idx in range(gmm_profile_z.shape[1]):
        val = gmm_profile_z.iloc[row_idx, col_idx]
        color = "white" if abs(val) >= 0.6 else "black"
        ax.text(col_idx, row_idx, f"{val:.2f}",
                ha="center", va="center", fontsize=8, color=color)
ax.set_xticks(np.arange(len(CLUSTER_FEATURES)))
ax.set_xticklabels(CLUSTER_FEATURES, rotation=45, ha="right")
ax.set_yticks(np.arange(len(gmm_profile_z.index)))
ax.set_yticklabels([RISK_TIER_NAMES[int(k)] for k in gmm_profile_z.index])
ax.set_title("Standardized Cluster Profile Heatmap (GMM, K=4)", fontweight="bold")
fig.colorbar(img, ax=ax, label="Z-score (within feature)")
plt.tight_layout()
plt.show()


#### Feature Means by GMM Cluster

In [ ]:
_PANEL_FEATURES = [
    ("temp_mean",       "Mean Temperature (°C)"),
    ("humidity_min",    "Minimum Humidity (%)"),
    ("wind_speed_max",  "Max Wind Speed (m/s)"),
]
_CLUSTER_LABELS = [0, 1, 2, 3]
_CLUSTER_COLORS = ["#2166ac", "#67a9cf", "#fdae61", "#b2182b"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (feat, ylabel) in zip(axes, _PANEL_FEATURES):
    means = df.groupby("gmm_label")[feat].mean()
    stds  = df.groupby("gmm_label")[feat].std()
    ax.bar(_CLUSTER_LABELS, means, yerr=stds,
           color=_CLUSTER_COLORS, edgecolor="black", linewidth=0.6,
           capsize=4, error_kw={"linewidth": 1})
    ax.set_xticks(_CLUSTER_LABELS)
    ax.set_xticklabels(RISK_TIER_NAMES, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(feat, fontweight="bold", fontsize=10)
    ax.grid(True, axis="y", alpha=0.25)

fig.suptitle("Feature Means by GMM Cluster (error bars = ±1 SD)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


#### Median FRP and Zone Multiplier by GMM Cluster

In [ ]:
_BAR_COLORS  = ["#dbe9f6", "#a6bddb", "#fdcc8a", "#e34a33"]
_baseline_idx = int(cluster_summary["median_frp"].idxmin())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(_CLUSTER_LABELS, cluster_summary["median_frp"],
            color=_BAR_COLORS, edgecolor="black")
for i, v in enumerate(cluster_summary["median_frp"]):
    axes[0].annotate(f"{v:.2f}", xy=(i, v), xytext=(0, 4),
                     textcoords="offset points", ha="center", va="bottom", fontsize=10)
axes[0].set_xticks(_CLUSTER_LABELS)
axes[0].set_xticklabels(RISK_TIER_NAMES)
axes[0].set_ylabel("Median FRP (MW, fire pixels only)")
axes[0].set_title("Median FRP by GMM Cluster", fontweight="bold")
axes[0].grid(True, axis="y", alpha=0.25)

axes[1].bar(_CLUSTER_LABELS, cluster_summary["zone_multiplier"],
            color=_BAR_COLORS, edgecolor="black")
for i, zm in enumerate(cluster_summary["zone_multiplier"]):
    axes[1].text(i, zm + 0.01, f"{zm:.2f}",
                 ha="center", va="bottom", fontsize=10)
axes[1].set_xticks(_CLUSTER_LABELS)
axes[1].set_xticklabels(RISK_TIER_NAMES)
axes[1].set_ylabel("Zone Multiplier")
axes[1].set_title("Zone Multiplier by GMM Cluster", fontweight="bold")
axes[1].grid(True, axis="y", alpha=0.25)
axes[1].text(0.98, 0.02,
             f"Baseline = {RISK_TIER_NAMES[_baseline_idx]} (1.00)",
             transform=axes[1].transAxes, ha="right", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()


#### Geographic Cluster Distribution

In [ ]:
from matplotlib.lines import Line2D

cluster_order = [0, 1, 2, 3]
colors_tab = sns.color_palette("tab10", n_colors=len(cluster_order))
color_map = dict(zip(cluster_order, colors_tab))

fig, ax = plt.subplots(figsize=(14, 8))
for c in cluster_order:
    grp = df[df["gmm_label"] == c]
    ax.scatter(grp["lon"], grp["lat"],
               s=1, alpha=0.2, color=color_map[c],
               label=f"Cluster {c} ({RISK_TIER_NAMES[c]})")

ax.set_title("Geographic Distribution of GMM Clusters", fontweight="bold")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
legend_handles = [
    Line2D([0], [0], marker="o", color="w",
           label=f"Cluster {c} — {RISK_TIER_NAMES[c]}",
           markerfacecolor=color_map[c], markersize=8)
    for c in cluster_order
]
ax.legend(handles=legend_handles, title="Cluster", loc="best")
plt.tight_layout()
plt.show()


#### FRP Distribution by Cluster (log scale)

In [ ]:
plot_df = df[df["frp"] > 0].copy()

plt.figure(figsize=(10, 5))
sns.boxplot(data=plot_df, x="gmm_label", y="frp",
            order=cluster_order, palette="Set2", showfliers=False)
plt.yscale("log")
plt.xticks(ticks=range(N_CLUSTERS), labels=RISK_TIER_NAMES)
plt.title("FRP Distribution by GMM Cluster (log scale)", fontweight="bold")
plt.xlabel("Risk Tier")
plt.ylabel("FRP (log scale, MW)")
plt.tight_layout()
plt.show()


### 6.7 Model Selection Justification

We choose **K=4** primarily by domain convention: wildfire danger communication is commonly framed as a four-tier risk scale (Low, Moderate, High, Extreme).

The monotonic decrease in GMM **BIC/AIC** with larger K is expected for meteorological data with continuous gradients, where additional components keep fitting finer structure.

The relatively low silhouette values are also expected because environmental transitions are gradual rather than sharply separated.

> **Spearman ρ = 1.0 interpretation:** The monotonic rank correlation between cluster label and mean FWI is **guaranteed by construction** of `remap_labels_by_fwi`, which explicitly reorders labels by ascending mean FWI. It serves as a sanity check, not as independent validation.